In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from black_scholes.pinn.bs_pinn_nd import BlackScholesMultiAssetPINN
from utility.model import ModelConfig, EarlyStopping

In [2]:
# Parameters

r = 0.1
sigmas = np.array([0.2, 0.3])
rho = 0.5
corr = np.array([
    [1.0, rho],
    [rho, 1.0]
])
K = 1.0
T = 1.0
S_mins = np.array([0.0, 0.0])
S_maxs = np.array([3.0, 3.0])

In [3]:
input_size = 3
hidden_sizes = [64, 64, 64]
output_size = 1
activation = nn.Sigmoid()
learning_rate = 0.001

# Scheduler
step_size = 2000
gamma = 0.7

model_config = ModelConfig(
    input_size=input_size,
    hidden_sizes=hidden_sizes,
    output_size=output_size,
    activation=activation,
    learning_rate=learning_rate,
    step_size=step_size,
    gamma=gamma
)

loss_weights = {
    'variational': 5,
    'terminal': 5,
    'Smax': 3,
    'Smin': 3
}

In [4]:
seed = 42
pinn = BlackScholesMultiAssetPINN(model_config, seed=seed)
pinn.set_params(K, r, sigmas, corr, T, S_mins, S_maxs)
pinn.set_loss_weights(loss_weights)

early_stopping = EarlyStopping(patience=500, min_delta=1e-7)
pinn.train(batch_size=4096, epochs=30000, early_stopping=early_stopping)

Iteration 0 | Training Loss: 0.3360867500305176 | Validation Loss: 0.29805880784988403
Iteration 500 | Training Loss: 0.05950511246919632 | Validation Loss: 0.05981229245662689
Iteration 1000 | Training Loss: 0.012257358059287071 | Validation Loss: 0.012265663594007492
Iteration 1500 | Training Loss: 0.008045350201427937 | Validation Loss: 0.008243294432759285
Iteration 2000 | Training Loss: 0.00639447383582592 | Validation Loss: 0.00658380426466465
Iteration 2500 | Training Loss: 0.005975812673568726 | Validation Loss: 0.005985911004245281
Iteration 3000 | Training Loss: 0.0056769829243421555 | Validation Loss: 0.005544893443584442
Iteration 3500 | Training Loss: 0.005220445804297924 | Validation Loss: 0.005161030683666468
Iteration 4000 | Training Loss: 0.004546199459582567 | Validation Loss: 0.004825816489756107
Iteration 4500 | Training Loss: 0.00436480063945055 | Validation Loss: 0.004568289965391159
Iteration 5000 | Training Loss: 0.004340062383562326 | Validation Loss: 0.0043266

In [5]:
pinn.save('../models/bs_pinn_nd/42.pth')